# Lyrics Analysis: Predicting Genre with Naive Bayes

In this notebook, we focus on **Multinomial Naive Bayes**, the industry standard for text analysis, to explore **Genre Classification**: Predicting a song's genre based on its lyrical themes.

The central hypothesis is that lyrics contain nuanced indicators of a song's genre, as different musical styles often employ distinct vocabularies and thematic elements.

### 1) Data Acquisition via KaggleHub

To ensure the project is reproducible and uses the most up-to-date information, the `kagglehub` library is used to programmatically download the datasets directly into the local environment.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

# Machine Learning tools
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Natural Language Processing (NLP) tools
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download the dataset
path = kagglehub.dataset_download("evabot/spotify-lyrics-dataset")
print("Path to dataset files:", path)

### 2) Loading Lyrics

Working with lyrics presents a distinct set of challenges. A robust loading logic handles different separators and missing data, prioritizing rows that contain actual lyrics.

In [ ]:
# Find the CSV file in the downloaded directory
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
lyrics_file_name = next((f for f in csv_files if 'lyrics' in f.lower()), csv_files[0])
lyrics_file_path = os.path.join(path, lyrics_file_name)

# Load the CSV with robust error handling
try:
    df_lyrics = pd.read_csv(lyrics_file_path, sep=';', decimal=',', on_bad_lines='skip')
    if 'lyrics' not in df_lyrics.columns and 'Lyrics' in df_lyrics.columns:
        df_lyrics = df_lyrics.rename(columns={'Lyrics': 'lyrics'})
    
    if 'lyrics' not in df_lyrics.columns:
        df_lyrics = pd.read_csv(lyrics_file_path, sep=',', decimal='.', on_bad_lines='skip')
        if 'lyrics' not in df_lyrics.columns and 'Lyrics' in df_lyrics.columns:
            df_lyrics = df_lyrics.rename(columns={'Lyrics': 'lyrics'})
except Exception as e:
    print(f"Error loading data: {e}")
    df_lyrics = pd.DataFrame()

# Drop rows where lyrics are missing
if not df_lyrics.empty and 'lyrics' in df_lyrics.columns:
    df_lyrics = df_lyrics.dropna(subset=['lyrics'])
    print(f"Loaded {len(df_lyrics)} songs from {lyrics_file_path}")
else:
    print("CRITICAL: Could not find 'lyrics' column.")

### 3) NLP Pipeline: Making Lyrics Machine-Readable

To assist the model, an extensive NLP cleaning pipeline was designed:
1. **Lowercase**: Ensures consistency.
2. **Regex Cleaning**: Removes technical metadata (like `[Chorus]`) and punctuation.
3. **Stop Word Removal**: Removes common words like "the", "is", etc.
4. **Lemmatization**: Reduces words to their base form (e.g., "dancing" to "dance").

In [ ]:
# Download necessary NLP resources
for resource in ['stopwords', 'wordnet', 'punkt', 'punkt_tab', 'omw-1.4']:
    nltk.download(resource, quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\(.*?\)', '', text)
    text = re.sub(r'\b(verse|chorus|bridge|outro|intro|hook|pre-chorus)\b\s*:', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = word_tokenize(text)
    processed_tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words and len(word) > 1
    ]

    return ' '.join(processed_tokens)

# Apply cleaning
df_lyrics['cleaned_lyrics'] = df_lyrics['lyrics'].apply(preprocess_text)
print("Sample of cleaned lyrics:")
print(df_lyrics[['lyrics', 'cleaned_lyrics']].head())

### 4) Classification Preparation: Vectorization

To feed lyrics into a model, **TF-IDF Vectorization** is used. A critical step is splitting the data **before** vectorization to prevent **data leakage**.

In [ ]:
def get_primary_genre(genre_list_str):
    if pd.isna(genre_list_str) or genre_list_str == '':
        return 'unknown'
    return genre_list_str.split(';')[0].strip()

df_lyrics['primary_genre'] = df_lyrics['genres'].apply(get_primary_genre)

# Identify genres that appear more than once for a safe train-test split
genre_counts = df_lyrics['primary_genre'].value_counts()
valid_genres = genre_counts[genre_counts > 1].index
print(f"Total genres identified: {len(genre_counts)}")
print(f"Valid genres (count > 1): {len(valid_genres)}")

### 5) Genre Classification with Multinomial Naive Bayes

Multinomial Naive Bayes is applied to the processed lyrics. This algorithm calculates the probability of a genre based on specific word frequencies.

In [ ]:
# Filter for valid genres
df_genre_task = df_lyrics[df_lyrics['primary_genre'].isin(valid_genres)].copy()

# Split the data
X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    df_genre_task['cleaned_lyrics'], 
    df_genre_task['primary_genre'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df_genre_task['primary_genre']
)

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.7)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Label Encoding
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_test = label_encoder.transform(y_test_raw)

# Train the model
model_genre = MultinomialNB()
model_genre.fit(X_train_tfidf, y_train)

# Evaluate performance
y_pred_genre = model_genre.predict(X_test_tfidf)
acc_genre = accuracy_score(y_test, y_pred_genre)
print(f"Genre Classification Accuracy: {acc_genre:.4f}")

### 6) Visualization of Results

In [ ]:
# Prepare data for the graph
comparison_data = [
    {'Task': 'Genre (Lyrics NB)', 'Accuracy': acc_genre}
]
df_comparison = pd.DataFrame(comparison_data)

# Create the bar chart
plt.figure(figsize=(8, 6))
sns.barplot(x='Task', y='Accuracy', data=df_comparison, palette='viridis', hue='Task', legend=False)

plt.title('Performance Summary: Naive Bayes Genre Prediction')
plt.ylim(0, 1)

for index, row in df_comparison.iterrows():
    plt.text(index, row['Accuracy'] + 0.02, f"{row['Accuracy']:.4f}", ha="center", fontweight='bold')

plt.show()

### 7) Summary of Findings

#### Summary:
- **Lyrical Genre Specificity**: The model demonstrates that lyrics are genre-specific, allowing for predictive modeling using Naive Bayes.
- **NLP Impact**: Proper preprocessing (cleaning, stop word removal, lemmatization) is crucial for making text data readable for machine learning models.

#### Limitations:
- **Contextual Loss**: Bag-of-words models like Naive Bayes ignore word order. Future iterations could use **N-grams** to capture more lyrical nuance.
- **Vocabulary Size**: The model is limited by the vocabulary present in the training set.